# Launch RAG API Server

這個 Notebook 用於啟動 OpenAI 相容的 API 伺服器，供 **OpenWebUI** 或其他前端應用程式串接。

**功能：**
- 啟動 FastAPI 服務
- 提供 `/v1/chat/completions` 接口
- 整合 RAG 檢索流程 (Two-Stage Retrieval)

**注意：**
- 執行最後一個 Cell 後，Kernel 會進入忙碌狀態（Blocking），直到您手動停止 (Stop) Kernel。
- 確保 Docker 的 Port 8000 有對外開放。

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. 設定專案路徑 (因為 notebook 在 notebooks/ 資料夾內)
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# 2. 載入環境變數
load_dotenv(repo_root / ".env")

print(f"Project Root: {repo_root}")

In [ ]:
# 3. 檢查設定是否正確
from rag_system.config import RAGConfig

try:
    config = RAGConfig.from_env()
    config.validate()
    print("✅ Configuration loaded successfully.")
    print(f"   - DB: {config.conn_string.split('@')[-1] if config.conn_string else 'Not Set'}")
    print(f"   - Embed Model: {config.embed_model}")
    print(f"   - Chat Model: {config.chat_model}")
except Exception as e:
    print(f"❌ Configuration Error: {e}")

## OpenWebUI 設定說明

當伺服器啟動後，請在 OpenWebUI 的 **Connections** 或 **Models** 設定中新增 OpenAI API 連線：

- **API Base URL**: `http://<你的Docker主機IP>:8000/v1` 
  - 如果 OpenWebUI 與此 Server 在同一個 Docker Network，可使用 `http://rag_jupyter:8000/v1`
- **API Key**: 任意字串 (例如 `sk-test`)，系統目前未強制驗證 Token。
- **Model ID**: `rag-agent` (API 預設回傳的模型名稱)

In [ ]:
import uvicorn
import nest_asyncio
from api import app

# 4. 解決 Jupyter 內執行 asyncio 的衝突
nest_asyncio.apply()

print("🚀 Starting Server on 0.0.0.0:8000...")
print("Press 'Stop' button in Jupyter to shutdown.")

# 5. 啟動伺服器
if __name__ == "__main__":
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )